# PyTorch: Binary Dataset Classification

In [ ]:
import torch
import numpy as np
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from torch import nn
from torch import optim
import matplotlib.pyplot as plt
from common import plot_loss_curves
from common.torch import get_summary_writer, get_default_device, set_seed

In [ ]:
set_seed()
device = get_default_device()

In [ ]:
print(f"PyTorch: version {torch.__version__}")
print(f"PyTorch: {device.type.upper()} device")

In [ ]:
# Hyperparameters
N_EPOCHS = 100
# Other parameters
N_SAMPLES = 1000
THRESHOLD = 0.6

## Prepare Datasets

In [ ]:
# Generate a dataset with two classes (red dots and blue dots)
x, y = make_circles(n_samples=N_SAMPLES,
                    factor=.5,
                    noise=.03,
                    random_state=0)

x.shape, y.shape

In [ ]:
plt.scatter(x[:, 0], x[:, 1], c=y, cmap="RdYlBu")

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
x_train = torch.from_numpy(x_train).float()
y_train = torch.from_numpy(y_train).float()
x_test = torch.from_numpy(x_test).float()
y_test = torch.from_numpy(y_test).float()

In [ ]:
print(f"Training dataset shape: {x_train.shape}, {y_train.shape}")
print(f"Test dataset shape: {x_test.shape}, {y_test.shape}")

## Define Model

In [ ]:
class CircleModel(nn.Module):
    def __init__(self, in_features: int = 2, out_features: int = 1):
        super().__init__()
        self.layer1 = nn.Linear(in_features, 16)
        self.layer2 = nn.Linear(16, 16)
        self.layer3 = nn.Linear(16, out_features)
        self.relu = nn.ReLU()


    def forward(self, x):
        z = self.layer1(x)
        z = self.relu(z)
        z = self.layer2(z)
        z = self.relu(z)
        y = self.layer3(z)
        return y

In [ ]:
model = CircleModel()

## Train Model

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), momentum=0.9, lr=0.1)

In [ ]:
# Get predictions using untrained model
with torch.inference_mode():
    y_log = model(x_test).squeeze()
    y_hat = (torch.sigmoid(y_log) > THRESHOLD).float()

In [ ]:
print(f"Input shape: {x_test.shape}")
print(f"Output shape: {y_hat.shape}")

print(f"Predictions: { y_hat[:10]}")
print(f"Targets: {y_test[:10]}")

# BCELoss expects probabilities (logits + activation, e.g. sigmoid)
# BCEWithLogitsLoss expects logits
print(f"Loss: {criterion(y_log , y_test)}")

In [ ]:
def accuracy(y_pred, y_true):
    tp = torch.eq(y_pred, y_true).sum().item()
    total = len(y_pred)
    acc = (tp / total) * 100
    return acc

### Training Loop

In [ ]:
model.to(device)

In [ ]:
x_train = x_train.to(device)
y_train = y_train.to(device)
x_test = x_test.to(device)
y_test = y_test.to(device)

In [ ]:
tr_losses = []
ts_losses = []
tr_accs = []
ts_accs = []

for epoch in range(N_EPOCHS):
    ## Training step
    model.train()
    y_log = model(x_train).squeeze()
    y_hat = (torch.sigmoid(y_log) > THRESHOLD).float()
    tr_loss = criterion(y_log, y_train)
    tr_accu = accuracy(y_hat, y_train)
    optimizer.zero_grad()
    tr_loss.backward()
    optimizer.step()

    ## Testing step
    model.eval()
    with torch.inference_mode():
        y_log = model(x_test).squeeze()
        y_hat = (torch.sigmoid(y_log) > THRESHOLD).float()
        ts_loss = criterion(y_log, y_test)
        ts_accu = accuracy(y_hat, y_test)

    if epoch % 10 == 0:
        tr_losses.append(tr_loss.item())
        ts_losses.append(ts_loss.item())
        tr_accs.append(tr_accu)
        ts_accs.append(ts_accu)
        print(f'Epoch: {epoch:02} | L/A: {tr_loss:.3f}/{tr_accu:5.1f} | Test L/A: {ts_loss:.3f}/{ts_accu:5.1f}')

In [ ]:
plot_loss_curves({
    "train_loss": tr_losses,
    "train_acc": tr_accs,
    "test_loss": ts_losses,
    "test_acc": ts_accs
})

## Evaluating

In [ ]:
def plot_decision_boundary(model: torch.nn.Module, x: torch.Tensor, y: torch.Tensor):
    """
    Plots decision boundaries of a model predicting on X in comparison to y
    """
    # Put everything to CPU (works better with NumPy + Matplotlib)
    model.to("cpu")
    x, y = x.to("cpu"), y.to("cpu")

    # Set up prediction boundaries and grid
    x_min, x_max = x[:, 0].min() - 0.1, x[:, 0].max() + 0.1
    y_min, y_max = x[:, 1].min() - 0.1, x[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 101), np.linspace(y_min, y_max, 101))

    # Make features
    X_to_pred_on = torch.from_numpy(np.column_stack((xx.ravel(), yy.ravel()))).float()

    # Make predictions
    model.eval()
    with torch.inference_mode():
        y_logits = model(X_to_pred_on)

    # Test for multi-class or binary and adjust logits to prediction labels
    if len(torch.unique(y)) > 2:
        y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)  # mutli-class
    else:
        y_pred = torch.round(torch.sigmoid(y_logits))  # binary

    # Reshape preds and plot
    y_pred = y_pred.reshape(xx.shape).detach().numpy()
    plt.contourf(xx, yy, y_pred, cmap="RdYlBu", alpha=0.7)
    plt.scatter(x[:, 0], x[:, 1], c=y, s=40, cmap="RdYlBu")
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())

In [ ]:
plot_decision_boundary(model, x_test, y_test)

In [ ]:
model.to(device)

# Get predictions using trained model
with torch.inference_mode():
    y_log = model(x_test).squeeze()
    y_hat = (torch.sigmoid(y_log) > THRESHOLD).float()

In [ ]:
print(f"Input shape: {x_test.shape}")
print(f"Output shape: {y_hat.shape}")
print(f"Predictions: { y_hat[:10]}")
print(f"Targets: {y_test[:10]}")
print(f"Loss: {criterion(y_log , y_test)}")